# Pattern Portal Real-Data Lab

A compact notebook companion for the Real-Data Cases page. This version is JupyterLite-compatible: it avoids packages that require native wheels and uses the small CSV files shipped with the site instead of remote dataset downloads.

## Install Dependencies
Run this cell first in JupyterLite. `yfinance` is intentionally excluded because its current dependency chain includes native/browser-incompatible wheels.

In [ ]:
%pip install pandas numpy scikit-learn matplotlib -q

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    root_mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

def read_portal_csv(path):
    """Read a Pattern Portal CSV in JupyterLite or local Jupyter."""
    site_path = path if path.startswith('/') else f'/{path}'
    try:
        open_url = __import__('pyodide.http', fromlist=['open_url']).open_url
        return pd.read_csv(open_url(site_path))
    except Exception:
        relative = site_path.lstrip('/')
        candidates = [
            Path(relative),
            Path('..') / relative,
            Path.cwd() / relative,
            Path.cwd().parent / relative,
        ]
        for candidate in candidates:
            if candidate.exists():
                return pd.read_csv(candidate)
        raise FileNotFoundError(f'Could not find {site_path} in JupyterLite or local paths.')

## Case 1: Housing Regression
Pipeline: load the local housing sample, split, train a baseline, report MAE/RMSE/R2, and inspect the largest errors.

In [ ]:
housing = read_portal_csv('/cases/datasets/housing_sample.csv')
X = housing.drop(columns=['median_house_value'])
y = housing['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE
)

model = HistGradientBoostingRegressor(random_state=RANDOM_STATE, max_iter=80)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print('MAE:', round(mean_absolute_error(y_test, pred), 4))
print('RMSE:', round(root_mean_squared_error(y_test, pred), 4))
print('R2:', round(r2_score(y_test, pred), 4))
pd.DataFrame({'actual': y_test, 'pred': pred, 'abs_error': np.abs(y_test - pred)}).sort_values('abs_error', ascending=False)

## Case 2: Fraud Classification
Use the local mini fraud sample to practice the workflow: split with class balance, fit a baseline, and inspect precision/recall behavior.

In [ ]:
fraud = read_portal_csv('/cases/datasets/fraud_sample.csv')
X = fraud.drop(columns=['is_fraud'])
y = fraud['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=RANDOM_STATE
)

model = RandomForestClassifier(
    n_estimators=80, class_weight='balanced', random_state=RANDOM_STATE
)
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.35).astype(int)

print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, digits=3, zero_division=0))
print('PR-AUC:', round(average_precision_score(y_test, proba), 4))

## Case 3: Time-Series Forecast
Use local energy-demand data: sort by time, create lag features from the past only, split by time, and compare against a naive baseline.

In [ ]:
daily = read_portal_csv('/cases/datasets/energy_demand_sample.csv')
daily['date'] = pd.to_datetime(daily['date'])
daily = daily.sort_values('date').set_index('date')

for lag in [1, 2, 3]:
    daily[f'lag_{lag}'] = daily['demand_kwh'].shift(lag)
daily['rolling_3'] = daily['demand_kwh'].shift(1).rolling(3).mean()
daily = daily.dropna()

split = int(len(daily) * 0.7)
train, test = daily.iloc[:split], daily.iloc[split:]
features = ['temp_c', 'is_weekend', 'lag_1', 'lag_2', 'lag_3', 'rolling_3']

model = RandomForestRegressor(n_estimators=80, random_state=RANDOM_STATE)
model.fit(train[features], train['demand_kwh'])
pred = model.predict(test[features])
naive = test['lag_1']

print('Model MAE:', round(mean_absolute_error(test['demand_kwh'], pred), 4))
print('Naive MAE:', round(mean_absolute_error(test['demand_kwh'], naive), 4))
pd.DataFrame({'actual': test['demand_kwh'], 'model': pred, 'naive': naive})

## Case 4: Market Backtest
JupyterLite cannot use `yfinance` reliably because live market clients often depend on native networking wheels. This cell uses the local OHLCV sample instead. Market indicators are heuristic until validated with point-in-time data, costs, slippage, and walk-forward testing.

In [ ]:
px = read_portal_csv('/cases/datasets/market_ohlcv_sample.csv')
px['date'] = pd.to_datetime(px['date'])
px = px.sort_values('date').set_index('date')
close = px['close']
returns = close.pct_change().fillna(0)

fast = close.rolling(3).mean()
slow = close.rolling(5).mean()
signal = (fast > slow).astype(int).shift(1).fillna(0)

turnover = signal.diff().abs().fillna(signal.abs())
cost = turnover * 0.0005
strategy = signal * returns - cost
equity = (1 + strategy).cumprod()
drawdown = equity / equity.cummax() - 1
sharpe = np.nan if strategy.std() == 0 else strategy.mean() / strategy.std() * np.sqrt(252)

print('Sharpe:', round(float(sharpe), 4) if not np.isnan(sharpe) else 'n/a')
print('Max drawdown:', round(float(drawdown.min()), 4))
print('Annual turnover:', round(float(turnover.mean() * 252), 4))
pd.DataFrame({'close': close, 'signal': signal, 'strategy': strategy, 'equity': equity})